In [0]:
#making changes
from pyspark.sql import functions as F
import uuid

# Source (sample data) and target (bronze layer)
SOURCE_SCHEMA = "samples.bakehouse"
TARGET_SCHEMA = "testing.bronze"

# One batch id per notebook run, shared across all tables loaded in this run
BATCH_ID = str(uuid.uuid4())

def load_bronze(source_table: str, target_table: str):
    """Read a source table as-is and append it to bronze with ingestion metadata columns."""
    src = f"{SOURCE_SCHEMA}.{source_table}"
    tgt = f"{TARGET_SCHEMA}.{target_table}"
    src_df = spark.read.table(src)
    df = (
        src_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_load_date", F.current_date())
        .withColumn("_source_system", F.lit("bakehouse_sample"))
        .withColumn("_source_table", F.lit(src))
        .withColumn("_batch_id", F.lit(BATCH_ID))
        .withColumn("_record_hash", F.sha2(F.to_json(F.struct(*src_df.columns)), 256))
    )
    (df.write.format("delta")
       .mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(tgt))
    print(f"Loaded {src} -> {tgt} (batch {BATCH_ID})")
    return tgt

In [0]:
load_bronze("sales_transactions", "bronze_sales_transactions")
display(spark.read.table("testing.bronze.bronze_sales_transactions").limit(5))

In [0]:
load_bronze("sales_customers", "bronze_customers")
display(spark.read.table("testing.bronze.bronze_customers").limit(5))

In [0]:
for src, tgt in [("sales_transactions", "bronze_sales_transactions"), ("sales_customers", "bronze_customers")]:
    src_cnt = spark.read.table(f"{SOURCE_SCHEMA}.{src}").count()
    tgt_cnt = spark.read.table(f"{TARGET_SCHEMA}.{tgt}").filter(F.col("_batch_id") == BATCH_ID).count()
    print(f"{tgt}: source={src_cnt}, bronze (this batch)={tgt_cnt}, match={src_cnt == tgt_cnt}")